# Demo 1 — Modelo Ingênuo de Próxima Palavra

## O que vamos fazer aqui?

Vamos construir um **modelo de linguagem do zero**, usando apenas Python puro e uma biblioteca de contagem.

Esse modelo vai:
1. Ler frases (um corpus)
2. Aprender quais palavras costumam aparecer depois de outras
3. Gerar novas frases baseado nesse aprendizado

---

## Por que isso é importante?

No fundo, **todo modelo de linguagem faz exatamente isso**:

> Dado o que veio antes, qual é a palavra (token) mais provável que vem a seguir?

A diferença entre este modelo e o GPT-4 não é o objetivo. O objetivo é o mesmo.
A diferença está em:
- **Como o contexto é representado** (aqui: só a palavra anterior; lá: toda a frase com atenção)
- **Escala** (aqui: dezenas de frases; lá: trilhões de tokens)
- **Arquitetura** (aqui: tabela de contagem; lá: transformer com bilhões de parâmetros)

---

## Conceito central: o Bigrama

Um **bigrama** é um par de palavras consecutivas.

Exemplo da frase `"o solo secou"` tem dois bigramas:
- `("o", "solo")`
- `("solo", "secou")`

Ao contar quantas vezes cada bigrama aparece no corpus, construímos uma tabela de probabilidades:

```
depois de "a produtividade"  →  80% das vezes vem "caiu", 20% vem "subiu"
depois de "o solo"           →  100% das vezes vem "secou"
```

Isso é um **modelo de linguagem de bigramas**.

---

> **Frase para guardar:**  
> *"Na essência, modelos de linguagem predizem o próximo token. O que muda radicalmente é a forma de representar contexto."*

---
## Parte 1 — Importações

Vamos usar apenas bibliotecas da biblioteca padrão do Python. Não precisamos de nenhuma dependência externa.


In [1]:
# re: módulo de expressões regulares — usado para limpar a pontuação do texto
import re

# collections.defaultdict: dicionário que cria valores padrão automaticamente
# quando acessamos uma chave que ainda não existe, ele cria com valor padrão
# usaremos defaultdict(list) para guardar: palavra -> [lista de próximas palavras]
from collections import defaultdict, Counter

# random: para sortear a próxima palavra de forma probabilística
import random

print("Bibliotecas carregadas com sucesso.")
print("Nenhuma dependência externa necessária.")

Bibliotecas carregadas com sucesso.
Nenhuma dependência externa necessária.


---
## Parte 2 — O Corpus

**Corpus** é o conjunto de textos que o modelo vai usar para aprender.

No caso dos LLMs reais, o corpus tem:
- Toda a Wikipedia
- Bilhões de páginas da internet
- Livros, artigos científicos, fóruns, código-fonte
- Dezenas de terabytes de texto

Aqui, nosso corpus tem **frases do agro brasileiro**.  
Isso é intencional: um corpus pequeno e temático torna o aprendizado do modelo **visível e verificável**.

Você conseguirá ver exatamente o que o modelo aprendeu — e por quê ele responde do jeito que responde.

In [2]:
# Este é o nosso corpus: um conjunto de frases sobre o agro brasileiro.
# Cada item da lista é uma frase completa.
#
# Nota: quanto mais frases com padrões repetidos, mais "confiante" o modelo
# vai ser ao prever a próxima palavra. Com poucas frases, ele terá muitas
# opções igualmente prováveis, o que também é didático de ver.

corpus = [
    # Frases sobre produtividade e clima
    "a produtividade do milho caiu por causa da estiagem",
    "a produtividade da soja subiu com as chuvas regulares",
    "a produtividade do café depende da temperatura e da umidade",
    "a produtividade caiu porque o solo estava seco",
    "a produtividade subiu depois das chuvas de outubro",

    # Frases sobre o solo
    "o solo seco prejudicou a lavoura de milho",
    "o solo precisa de nutrientes para produzir bem",
    "o solo fértil garante boa produtividade ao produtor",
    "o solo foi preparado antes do plantio do milho",

    # Frases sobre estiagem e chuva
    "a estiagem prolongada afetou a lavoura de soja",
    "a estiagem causou perdas na produção de milho",
    "a estiagem reduziu a produtividade do campo",
    "a chuva regular beneficiou a lavoura de café",
    "a chuva chegou tarde e prejudicou o plantio",
    "a chuva forte causou alagamento na fazenda",

    # Frases sobre lavoura e plantio
    "a lavoura de milho precisa de irrigação constante",
    "a lavoura de soja foi afetada pela geada",
    "a lavoura de café exige cuidado com a temperatura",
    "o plantio foi feito no inicio da estação chuvosa",
    "o plantio do milho depende da umidade do solo",

    # Frases sobre o produtor
    "o produtor rural investe em tecnologia para aumentar a produtividade",
    "o produtor rural depende das chuvas para a safra",
    "o produtor visitou a fazenda depois da chuva",
    "o produtor precisa monitorar o solo constantemente",

    # Frases sobre o agro
    "o agro brasileiro depende de boas condições climáticas",
    "o agro é responsável por grande parte das exportações",
    "o agro precisa de inovação para competir no mercado global",
]

print(f"Corpus criado com {len(corpus)} frases.")
print()
print("Algumas frases de exemplo:")
for frase in corpus[:5]:
    print(f"  → {frase}")

Corpus criado com 27 frases.

Algumas frases de exemplo:
  → a produtividade do milho caiu por causa da estiagem
  → a produtividade da soja subiu com as chuvas regulares
  → a produtividade do café depende da temperatura e da umidade
  → a produtividade caiu porque o solo estava seco
  → a produtividade subiu depois das chuvas de outubro


---
## Parte 3 — Pré-processamento e Tokenização

Antes de contar bigramas, precisamos transformar o texto em tokens.

**Token** = a unidade mínima que o modelo processa. Pode ser:
- Uma palavra inteira (como faremos aqui)
- Um pedaço de palavra (como GPT usa — ex: "produtividade" vira `["produt", "ividade"]`)
- Um caractere
- Um byte

Aqui vamos usar **palavras** como tokens. É o mais simples e intuitivo.

O pré-processamento que faremos:
1. Converter tudo para minúsculas ("O" e "o" são a mesma palavra)
2. Remover pontuação ("milho," e "milho" devem ser o mesmo token)
3. Dividir a frase em lista de palavras

In [3]:
def tokenizar(frase):
    """
    Transforma uma frase em uma lista de tokens (palavras).

    Passos:
    1. Converter para minúsculas: evita que "Milho" e "milho" sejam tokens diferentes
    2. Remover pontuação: evita que "milho," e "milho" sejam tokens diferentes
    3. Dividir por espaço: cada palavra vira um token
    4. Filtrar strings vazias: evita tokens fantasmas gerados por espaços duplos

    Args:
        frase (str): Uma frase de texto corrido.

    Returns:
        list[str]: Lista de tokens (palavras limpas).

    Exemplo:
        tokenizar("A Produtividade caiu!") → ["a", "produtividade", "caiu"]
    """
    # Passo 1: minúsculas
    frase = frase.lower()

    # Passo 2: remove tudo que não seja letra, número ou espaço
    # O padrão r'[^a-záàâãéêíóôõúüç\s]' mantém letras com acento e espaços
    frase = re.sub(r'[^a-záàâãéêíóôõúüç\s]', '', frase)

    # Passo 3 e 4: divide por espaço e remove strings vazias
    tokens = [palavra for palavra in frase.split() if palavra]

    return tokens


# Vamos testar a função com alguns exemplos
print("=== Testando a tokenização ===")
print()

exemplos = [
    "A Produtividade caiu!",
    "o solo, seco e árido, prejudicou a lavoura.",
    corpus[0],
]

for ex in exemplos:
    tokens = tokenizar(ex)
    print(f"Entrada : {ex!r}")
    print(f"Tokens  : {tokens}")
    print()

=== Testando a tokenização ===

Entrada : 'A Produtividade caiu!'
Tokens  : ['a', 'produtividade', 'caiu']

Entrada : 'o solo, seco e árido, prejudicou a lavoura.'
Tokens  : ['o', 'solo', 'seco', 'e', 'árido', 'prejudicou', 'a', 'lavoura']

Entrada : 'a produtividade do milho caiu por causa da estiagem'
Tokens  : ['a', 'produtividade', 'do', 'milho', 'caiu', 'por', 'causa', 'da', 'estiagem']



---
## Parte 4 — Construindo o Modelo de Bigramas

Agora vem o coração do modelo: **aprender com o corpus**.

O que vamos fazer:
- Para cada frase, extraímos todos os bigramas
- Para cada bigrama `(palavra_A, palavra_B)`, registramos que "depois de `palavra_A`, a palavra `palavra_B` apareceu"
- Guardamos isso num dicionário: `{ palavra_A: [palavra_B1, palavra_B2, ...] }`

Se `palavra_B` aparece 3 vezes depois de `palavra_A`, ela vai aparecer 3 vezes na lista, o que naturalmente cria probabilidade maior de ser sorteada.

**Isso é aprendizado de máquina em sua forma mais simples:** observar padrões nos dados e registrá-los.

In [12]:
def construir_modelo_bigramas(corpus):
    """
    Aprende um modelo de bigramas a partir de um corpus de frases.

    Para cada par de palavras consecutivas (bigrama) encontrado no corpus,
    registra que a segunda palavra pode vir depois da primeira.

    Estrutura do modelo:
        modelo = {
            "palavra_anterior": ["proxima1", "proxima2", "proxima1", ...],
            ...
        }

    Palavras que aparecem mais vezes na lista têm mais chance de ser sorteadas,
    o que naturalmente representa a frequência observada no corpus.

    Args:
        corpus (list[str]): Lista de frases de texto.

    Returns:
        defaultdict(list): O modelo de bigramas.
    """
    # defaultdict(list): quando acessamos uma chave nova, ela começa com []
    # Sem isso, precisaríamos checar "if chave not in modelo" a cada passo
    modelo = defaultdict(list)

    for frase in corpus:
        # Tokenizamos a frase
        tokens = tokenizar(frase)

        # Precisamos de ao menos 2 tokens para ter um bigrama
        if len(tokens) < 2:
            continue

        # zip(tokens, tokens[1:]) cria pares consecutivos:
        # tokens       = ["a", "produtividade", "caiu"]
        # tokens[1:]   = [     "produtividade", "caiu", ...]
        # pares = [("a", "produtividade"), ("produtividade", "caiu")]
        for palavra_atual, proxima_palavra in zip(tokens, tokens[1:]):
            # Registra: "depois de palavra_atual, apareceu proxima_palavra"
            modelo[palavra_atual].append(proxima_palavra)

    return modelo


# Treinamos o modelo com nosso corpus
modelo = construir_modelo_bigramas(corpus)

print("Modelo treinado!")
print(f"Número de palavras no vocabulário do modelo: {len(modelo)}")
print()
print("Palavras conhecidas pelo modelo:")
print(sorted(modelo.keys()))

Modelo treinado!
Número de palavras no vocabulário do modelo: 84

Palavras conhecidas pelo modelo:
['a', 'afetada', 'afetou', 'agro', 'alagamento', 'antes', 'ao', 'as', 'aumentar', 'beneficiou', 'boa', 'boas', 'brasileiro', 'café', 'caiu', 'causa', 'causou', 'chegou', 'chuva', 'chuvas', 'com', 'competir', 'condições', 'cuidado', 'da', 'das', 'de', 'depende', 'depois', 'do', 'e', 'em', 'estava', 'estação', 'estiagem', 'exige', 'fazenda', 'feito', 'foi', 'forte', 'fértil', 'garante', 'grande', 'inicio', 'inovação', 'investe', 'irrigação', 'lavoura', 'mercado', 'milho', 'monitorar', 'na', 'no', 'nutrientes', 'o', 'para', 'parte', 'pela', 'perdas', 'plantio', 'por', 'porque', 'precisa', 'prejudicou', 'preparado', 'produtividade', 'produtor', 'produzir', 'produção', 'prolongada', 'reduziu', 'regular', 'responsável', 'rural', 'seco', 'soja', 'solo', 'subiu', 'tarde', 'tecnologia', 'temperatura', 'umidade', 'visitou', 'é']


---
## Parte 5 — Inspecionando o que o modelo aprendeu

Vamos olhar dentro do modelo para entender o que ele "sabe".

Isso é algo que **não conseguimos fazer com LLMs reais**, pois eles têm bilhões de parâmetros espalhados em matrizes numéricas, impossíveis de inspecionar diretamente.

Aqui, o conhecimento está explícito e legível.

In [13]:
def mostrar_o_que_modelo_aprendeu(modelo, palavra, top_n=10):
    """
    Mostra quais palavras o modelo associa como 'próxima palavra' após a palavra dada,
    com suas frequências e probabilidades.

    Args:
        modelo: O modelo de bigramas.
        palavra (str): A palavra que queremos investigar.
        top_n (int): Quantas opções mostrar.
    """
    if palavra not in modelo:
        print(f"A palavra '{palavra}' não foi vista no corpus.")
        return

    # Lista de todas as palavras que o modelo viu depois de 'palavra'
    candidatos = modelo[palavra]
    total = len(candidatos)

    # Counter conta quantas vezes cada palavra aparece na lista
    contagem = Counter(candidatos)

    print(f"Depois da palavra '{palavra}' ({total} ocorrências no corpus):")
    print(f"{'Próxima palavra':<20} {'Contagem':>10} {'Probabilidade':>15}")
    print("-" * 48)

    # most_common(top_n) retorna as N palavras mais frequentes
    for proxima, contagem_n in contagem.most_common(top_n):
        probabilidade = contagem_n / total
        barra = "█" * int(probabilidade * 20)  # barra visual de probabilidade
        print(f"{proxima:<20} {contagem_n:>10} {probabilidade:>14.1%}  {barra}")


# Vamos inspecionar algumas palavras-chave do nosso corpus
print("=" * 60)
mostrar_o_que_modelo_aprendeu(modelo, "a")

print()
print("=" * 60)
mostrar_o_que_modelo_aprendeu(modelo, "produtividade")

print()
print("=" * 60)
mostrar_o_que_modelo_aprendeu(modelo, "o")

print()
print("=" * 60)
mostrar_o_que_modelo_aprendeu(modelo, "solo")

Depois da palavra 'a' (22 ocorrências no corpus):
Próxima palavra        Contagem   Probabilidade
------------------------------------------------
produtividade                 7          31.8%  ██████
lavoura                       6          27.3%  █████
estiagem                      3          13.6%  ██
chuva                         3          13.6%  ██
temperatura                   1           4.5%  
safra                         1           4.5%  
fazenda                       1           4.5%  

Depois da palavra 'produtividade' (7 ocorrências no corpus):
Próxima palavra        Contagem   Probabilidade
------------------------------------------------
do                            3          42.9%  ████████
da                            1          14.3%  ██
caiu                          1          14.3%  ██
subiu                         1          14.3%  ██
ao                            1          14.3%  ██

Depois da palavra 'o' (16 ocorrências no corpus):
Próxima palavra        C

---
## Parte 6 — Gerando Frases

Agora vamos usar o modelo para **gerar novas frases**.

O processo é simples:
1. Começa com uma palavra-semente
2. Olha no modelo quais palavras costumam vir depois dessa palavra
3. Sorteia uma delas (de forma proporcional à frequência)
4. A palavra sorteada vira a nova "palavra atual"
5. Repete até atingir o tamanho desejado

Isso é exatamente o que um LLM faz — só que em escala e com arquitetura muito mais sofisticada.

> **Nota importante:** `random.choice(lista)` escolhe um elemento aleatoriamente com probabilidade uniforme. Como palavras frequentes aparecem mais vezes na lista, elas são naturalmente mais prováveis. Isso é chamado de **amostragem proporcional à frequência**.

In [6]:
def gerar_frase(modelo, palavra_inicial, tamanho_maximo=15):
    """
    Gera uma frase usando o modelo de bigramas.

    Algoritmo:
    1. Começa com palavra_inicial
    2. Sorteia a próxima palavra a partir das opções que o modelo conhece
    3. Repete até atingir tamanho_maximo ou não haver próxima palavra

    Args:
        modelo: O modelo de bigramas (dict de listas).
        palavra_inicial (str): A palavra com que a frase começa.
        tamanho_maximo (int): Número máximo de palavras a gerar.

    Returns:
        str: A frase gerada.
    """
    # Verificamos se a palavra inicial é conhecida pelo modelo
    if palavra_inicial not in modelo:
        return f"[Palavra '{palavra_inicial}' não está no vocabulário do modelo]"

    # Começamos a frase com a palavra inicial
    frase = [palavra_inicial]

    # A palavra atual é a última palavra da frase até agora
    palavra_atual = palavra_inicial

    for _ in range(tamanho_maximo - 1):
        # Verifica se o modelo conhece continuações para a palavra atual
        if palavra_atual not in modelo:
            # Chegamos a um beco sem saída — o modelo nunca viu essa palavra
            # no meio de uma frase, só no final
            break

        # Lista de palavras que podem vir depois da palavra atual
        candidatos = modelo[palavra_atual]

        # Sorteamos uma palavra candidata
        # random.choice faz isso com distribuição uniforme na lista,
        # mas como palavras frequentes aparecem mais na lista, elas
        # têm mais chance de ser sorteadas — isso é amostragem por frequência
        proxima_palavra = random.choice(candidatos)

        # Adicionamos a palavra à frase
        frase.append(proxima_palavra)

        # A próxima palavra passa a ser a palavra atual para o próximo passo
        palavra_atual = proxima_palavra

    # Juntamos a lista de palavras em uma string
    return " ".join(frase)


# Fixamos a semente aleatória para resultados reproduzíveis
# (sem isso, cada execução gera frases diferentes)
random.seed(42)

print("=== Gerando frases com o modelo ===")
print()

# Vamos gerar 5 frases para cada palavra inicial
palavras_iniciais = ["a", "o", "a", "o", "a"]

for i, palavra in enumerate(palavras_iniciais, 1):
    frase = gerar_frase(modelo, palavra, tamanho_maximo=12)
    print(f"Frase {i}: {frase}")

=== Gerando frases com o modelo ===

Frase 1: a safra
Frase 2: o solo estava seco prejudicou a produtividade ao produtor rural depende da
Frase 3: a produtividade do café depende da estação chuvosa
Frase 4: o agro brasileiro depende das chuvas regulares
Frase 5: a chuva chegou tarde e da chuva chegou tarde e prejudicou a


---
## Parte 7 — Gerando com palavras-semente mais específicas

Agora vamos ver como o modelo responde a diferentes pontos de partida.

Palavras mais específicas geram frases mais previsíveis — porque o modelo viu poucas continuações possíveis para elas.

In [7]:
random.seed(0)  # semente para reprodutibilidade

# Mapeamos palavras iniciais e quantas frases gerar para cada
experimentos = [
    ("produtividade", 4),
    ("estiagem",      4),
    ("lavoura",       4),
    ("solo",          4),
    ("produtor",      4),
    ("agro",          4),
]

for palavra_inicial, n_frases in experimentos:
    print(f"\n--- Iniciando com '{palavra_inicial}' ---")
    for i in range(n_frases):
        frase = gerar_frase(modelo, palavra_inicial, tamanho_maximo=10)
        print(f"  {i+1}. {frase}")


--- Iniciando com 'produtividade' ---
  1. produtividade do milho precisa de milho depende de milho precisa
  2. produtividade caiu porque o solo precisa de milho depende das
  3. produtividade subiu com a produtividade ao produtor visitou a lavoura
  4. produtividade do plantio foi afetada pela geada

--- Iniciando com 'estiagem' ---
  1. estiagem causou alagamento na produção de inovação para produzir bem
  2. estiagem causou perdas na produção de café depende da chuva
  3. estiagem prolongada afetou a produtividade do campo
  4. estiagem causou perdas na fazenda depois da chuva forte causou

--- Iniciando com 'lavoura' ---
  1. lavoura de soja foi afetada pela geada
  2. lavoura de café depende das chuvas regulares
  3. lavoura de outubro
  4. lavoura de boas condições climáticas

--- Iniciando com 'solo' ---
  1. solo estava seco prejudicou a produtividade do milho depende de
  2. solo constantemente
  3. solo foi afetada pela geada
  4. solo constantemente

--- Iniciando com 'pro

---
## Parte 8 — Visualizando as Probabilidades

Vamos criar uma visualização para entender **o que o modelo "sabe"** sobre uma palavra.

Para cada palavra de entrada, mostramos um ranking das prováveis continuações.

In [8]:
def visualizar_probabilidades(modelo, palavra, max_opcoes=8):
    """
    Exibe uma visualização em texto das probabilidades de cada próxima palavra
    dado uma palavra de entrada.

    Args:
        modelo: O modelo de bigramas.
        palavra (str): A palavra de entrada.
        max_opcoes (int): Número máximo de opções a mostrar.
    """
    if palavra not in modelo:
        print(f"'{palavra}' não está no vocabulário.")
        return

    candidatos = modelo[palavra]
    total = len(candidatos)
    contagem = Counter(candidatos)

    print(f"\nDistribuição de próximas palavras para: '{palavra}'")
    print(f"(baseada em {total} ocorrências no corpus)")
    print()

    for proxima, n in contagem.most_common(max_opcoes):
        prob = n / total
        # Barra proporcional à probabilidade (máx 40 caracteres)
        largura_barra = int(prob * 40)
        barra = "▓" * largura_barra + "░" * (40 - largura_barra)
        print(f"  '{proxima:<18}' {barra} {prob:.0%}")


# Visualizamos para palavras-chave do nosso domínio
for palavra in ["produtividade", "a", "o", "lavoura"]:
    visualizar_probabilidades(modelo, palavra)
    print()


Distribuição de próximas palavras para: 'produtividade'
(baseada em 7 ocorrências no corpus)

  'do                ' ▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓░░░░░░░░░░░░░░░░░░░░░░░ 43%
  'da                ' ▓▓▓▓▓░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░ 14%
  'caiu              ' ▓▓▓▓▓░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░ 14%
  'subiu             ' ▓▓▓▓▓░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░ 14%
  'ao                ' ▓▓▓▓▓░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░ 14%


Distribuição de próximas palavras para: 'a'
(baseada em 22 ocorrências no corpus)

  'produtividade     ' ▓▓▓▓▓▓▓▓▓▓▓▓░░░░░░░░░░░░░░░░░░░░░░░░░░░░ 32%
  'lavoura           ' ▓▓▓▓▓▓▓▓▓▓░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░ 27%
  'estiagem          ' ▓▓▓▓▓░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░ 14%
  'chuva             ' ▓▓▓▓▓░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░ 14%
  'temperatura       ' ▓░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░ 5%
  'safra             ' ▓░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░ 5%
  'fazenda           ' ▓░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░ 5%


Distr

---
## Parte 9 — O Limite: contexto de 1 palavra

Vamos ver um problema fundamental do modelo de bigramas:

> **Ele só olha para 1 palavra antes. Não tem memória de contexto.**

Isso significa que a palavra `"caiu"` na frase:
- `"a produtividade caiu..."` → vem depois de "produtividade"
- `"o preço caiu..."` → vem depois de "preço"

Mas nosso modelo não sabe **por que** caiu. Não conecta "estiagem" com "caiu" se elas estão separadas por outras palavras.

Este é exatamente o problema que os **trigramas** tentam resolver (parcialmente) e que o **transformer** resolve de verdade.

In [9]:
# Demonstrando a falta de memória contextual
#
# O modelo só sabe que depois de "caiu" podem vir certas palavras.
# Ele não sabe SE vamos continuar com "porque a estiagem" ou "mas o clima melhorou".
# Tudo depende apenas da última palavra — sem nenhum contexto anterior.

print("=== Demonstrando a limitação de memória ===")
print()
print("O modelo toma a mesma decisão independente do que veio antes.")
print("Para ele, só importa a ÚLTIMA palavra.")
print()

# Vamos gerar múltiplas frases começando da mesma palavra
# para ver como o modelo escolhe caminhos diferentes sem nenhum critério semântico
random.seed(100)

print("10 frases geradas a partir de 'a':")
print()
for i in range(10):
    frase = gerar_frase(modelo, "a", tamanho_maximo=8)
    print(f"  {i+1:>2}. {frase}")

print()
print("Observe: o modelo gera frases diferentes porque sorteia a próxima")
print("palavra probabilisticamente, mas sem nenhuma coerência semântica de longo prazo.")

=== Demonstrando a limitação de memória ===

O modelo toma a mesma decisão independente do que veio antes.
Para ele, só importa a ÚLTIMA palavra.

10 frases geradas a partir de 'a':

   1. a produtividade caiu porque o solo constantemente
   2. a chuva chegou tarde e da umidade do
   3. a estiagem prolongada afetou a lavoura de soja
   4. a lavoura de soja foi afetada pela geada
   5. a lavoura de soja foi afetada pela geada
   6. a lavoura de café exige cuidado com as
   7. a lavoura de boas condições climáticas
   8. a safra
   9. a lavoura de café depende da umidade do
  10. a temperatura e prejudicou o agro é responsável

Observe: o modelo gera frases diferentes porque sorteia a próxima
palavra probabilisticamente, mas sem nenhuma coerência semântica de longo prazo.


---
## Parte 10 — Bônus: Modelo de Trigramas

Um **trigrama** usa as **2 palavras anteriores** como contexto (em vez de 1).

Isso melhora a coerência local, mas **não resolve** o problema de dependências longas.

Compare:
- Bigrama: `"solo" → "seco"`
- Trigrama: `"o solo" → "seco"` (mais específico, mais coerente localmente)

O mesmo princípio vale para 4-gramas, 5-gramas... mas a quantidade de dados necessária cresce exponencialmente. E mesmo assim, **o contexto sempre terá um limite fixo**.

É por isso que o transformer é uma mudança de paradigma: ele não tem limite fixo de contexto.

In [10]:
def construir_modelo_trigramas(corpus):
    """
    Aprende um modelo de trigramas a partir do corpus.

    Aqui, o contexto é formado pelas 2 últimas palavras (em vez de 1).
    A chave do dicionário é uma TUPLA de 2 palavras.

    Estrutura:
        modelo = {
            ("palavra_1", "palavra_2"): ["proxima1", "proxima2", ...],
            ...
        }
    """
    modelo = defaultdict(list)

    for frase in corpus:
        tokens = tokenizar(frase)

        # Precisamos de ao menos 3 tokens para ter um trigrama
        if len(tokens) < 3:
            continue

        # zip com 3 listas defasadas: (w0,w1,w2), (w1,w2,w3), ...
        for w1, w2, proxima in zip(tokens, tokens[1:], tokens[2:]):
            # A chave é a tupla (w1, w2) — as 2 palavras de contexto
            modelo[(w1, w2)].append(proxima)

    return modelo


def gerar_frase_trigrama(modelo, palavra1, palavra2, tamanho_maximo=15):
    """
    Gera uma frase usando o modelo de trigramas.

    Args:
        modelo: O modelo de trigramas.
        palavra1 (str): Primeira palavra da frase.
        palavra2 (str): Segunda palavra da frase.
        tamanho_maximo (int): Número máximo de palavras.

    Returns:
        str: A frase gerada.
    """
    frase = [palavra1, palavra2]
    contexto = (palavra1, palavra2)

    for _ in range(tamanho_maximo - 2):
        if contexto not in modelo:
            break  # beco sem saída

        candidatos = modelo[contexto]
        proxima = random.choice(candidatos)

        frase.append(proxima)

        # Avançamos a janela de contexto: descartamos a palavra mais antiga
        # e adicionamos a que acabamos de gerar
        contexto = (contexto[1], proxima)

    return " ".join(frase)


# Treinamos o modelo de trigramas
modelo_trigrama = construir_modelo_trigramas(corpus)

print(f"Modelo de trigramas treinado.")
print(f"Contextos únicos aprendidos: {len(modelo_trigrama)}")
print()

# Comparação: bigrama vs trigrama
random.seed(7)

print("=== Bigrama (contexto: 1 palavra) ===")
for i in range(5):
    frase = gerar_frase(modelo, "a", tamanho_maximo=10)
    print(f"  {i+1}. {frase}")

print()
print("=== Trigrama (contexto: 2 palavras) ===")
for i in range(5):
    frase = gerar_frase_trigrama(modelo_trigrama, "a", "produtividade", tamanho_maximo=10)
    print(f"  {i+1}. {frase}")

print()
print("Observe: o trigrama gera frases mais coerentes localmente,")
print("mas ainda falha em dependências longas.")

Modelo de trigramas treinado.
Contextos únicos aprendidos: 134

=== Bigrama (contexto: 1 palavra) ===
  1. a produtividade da umidade do milho depende da temperatura e
  2. a lavoura de outubro
  3. a produtividade caiu porque o plantio do campo
  4. a chuva regular beneficiou a safra
  5. a safra

=== Trigrama (contexto: 2 palavras) ===
  1. a produtividade subiu depois das chuvas de outubro
  2. a produtividade subiu depois das chuvas de outubro
  3. a produtividade subiu depois das chuvas de outubro
  4. a produtividade do campo
  5. a produtividade do milho caiu por causa da estiagem

Observe: o trigrama gera frases mais coerentes localmente,
mas ainda falha em dependências longas.


---
## Parte 11 — Resumo do que construímos

Neste notebook, construímos um **modelo de linguagem de bigramas** do zero.

Vimos que ele:
- **Aprende** padrões de co-ocorrência de palavras
- **Gera** frases sorteando a próxima palavra probabilisticamente
- **Falha** quando precisa de memória de contexto mais longo

---

## O que muda nos LLMs reais?

| Aspecto | Modelo Ingênuo (aqui) | LLM (GPT, Claude, etc.) |
|---|---|---|
| **Contexto** | Última 1-2 palavras | Toda a conversa (milhares de tokens) |
| **Representação** | Tabela de contagem | Vetores densos (embeddings) |
| **Arquitetura** | Dicionário de listas | Transformer com bilhões de parâmetros |
| **Corpus** | ~27 frases | Trilhões de tokens |
| **Treinamento** | Contagem direta | Gradiente descendente em GPUs |
| **Objetivo** | Idêntico | Idêntico |

---

> **Frase para guardar:**  
> *"LLMs continuam fazendo previsão token a token. O que muda radicalmente é a forma de representar contexto."*

---

## Próximo notebook

No **Demo 2**, vamos ver exatamente onde esse modelo falha: em frases com dependências longas.  
Vamos comparar janelas de contexto curtas e longas, e tornar visível **por que** o transformer foi necessário.

In [11]:
# Célula final: resumo quantitativo do modelo

print("=== Resumo do modelo treinado ===")
print()

# Contando o total de tokens no corpus
todos_tokens = []
for frase in corpus:
    todos_tokens.extend(tokenizar(frase))

vocabulario = set(todos_tokens)
total_bigramas = sum(len(v) for v in modelo.values())

print(f"Frases no corpus         : {len(corpus)}")
print(f"Total de tokens          : {len(todos_tokens)}")
print(f"Vocabulário único        : {len(vocabulario)} palavras")
print(f"Bigramas registrados     : {total_bigramas}")
print(f"Palavras com continuações: {len(modelo)}")
print()
print("Top 10 palavras mais frequentes no corpus:")
for palavra, count in Counter(todos_tokens).most_common(10):
    print(f"  {palavra:<20} {count:>4}x")

=== Resumo do modelo treinado ===

Frases no corpus         : 27
Total de tokens          : 227
Vocabulário único        : 96 palavras
Bigramas registrados     : 200
Palavras com continuações: 84

Top 10 palavras mais frequentes no corpus:
  a                      22x
  o                      16x
  de                     12x
  produtividade           8x
  do                      7x
  da                      7x
  solo                    7x
  milho                   6x
  lavoura                 6x
  produtor                5x
